# Clase 17: Estadística inferencial — pruebas de hipótesis

Las pruebas de hipótesis nos dicen si una diferencia que vemos en los datos
es real o simplemente producto del azar.

**Datasets:** `transporte.csv` y `estudiantes.csv`

**Pregunta guía:** ¿El retraso del transporte es realmente mayor en días lluviosos,
o esa diferencia puede deberse al azar?

In [ ]:
# Qué hace: importar librerías y cargar los datasets de la clase
# Por qué sirve: scipy.stats contiene todas las pruebas estadísticas que usaremos
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")

transporte = pd.read_csv("../../datasets/transporte.csv")
estudiantes = pd.read_csv("../../datasets/estudiantes.csv")

print("Transporte:", transporte.shape)
print(transporte.dtypes)
print()
print("Estudiantes:", estudiantes.shape)
print(estudiantes.dtypes)

In [ ]:
# Qué hace: exploración descriptiva de los grupos antes de hacer cualquier prueba
# Por qué sirve: siempre hay que entender los datos antes de aplicar estadística inferencial

print("=== ESTADÍSTICA DESCRIPTIVA POR GRUPO (lluvia) ===")
resumen = transporte.groupby("lluvia")["retraso_min"].agg(
    n="count",
    media="mean",
    mediana="median",
    desviacion_std="std",
    minimo="min",
    maximo="max"
).round(2)
print(resumen)

# Separar los grupos para los análisis siguientes
con_lluvia = transporte[transporte["lluvia"] == True]["retraso_min"].dropna()
sin_lluvia = transporte[transporte["lluvia"] == False]["retraso_min"].dropna()

diferencia_medias = con_lluvia.mean() - sin_lluvia.mean()
print(f"\nDiferencia de medias observada: {diferencia_medias:.2f} minutos")

In [ ]:
# Qué hace: visualizar la diferencia entre grupos antes de hacer la prueba formal
# Por qué sirve: un gráfico siempre debe acompañar a la prueba estadística

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot: compara medianas y dispersión entre grupos
sns.boxplot(data=transporte, x="lluvia", y="retraso_min",
            palette=["lightblue", "steelblue"], ax=axes[0])
axes[0].set_title("Retraso por condición de lluvia")
axes[0].set_xlabel("¿Hay lluvia?")
axes[0].set_ylabel("Retraso (minutos)")
axes[0].set_xticklabels(["Sin lluvia", "Con lluvia"])

# Violinplot: muestra la distribución completa
sns.violinplot(data=transporte, x="lluvia", y="retraso_min",
               palette=["lightblue", "steelblue"], inner="quartile", ax=axes[1])
axes[1].set_title("Distribución del retraso por lluvia")
axes[1].set_xlabel("¿Hay lluvia?")
axes[1].set_ylabel("Retraso (minutos)")
axes[1].set_xticklabels(["Sin lluvia", "Con lluvia"])

plt.tight_layout()
plt.show()

In [ ]:
# Qué hace: formular y ejecutar la prueba t de Student para dos grupos independientes
# Por qué sirve: responde si la diferencia de medias es estadísticamente significativa

# --- FORMULACIÓN DE HIPÓTESIS ---
# H0 (hipótesis nula):        La media de retraso con lluvia = media sin lluvia
#                             No hay diferencia real entre los dos grupos.
# H1 (hipótesis alternativa): La media de retraso con lluvia != media sin lluvia
#                             Hay una diferencia real entre los grupos.

# Prueba t para grupos independientes
estadistico_t, p_value = stats.ttest_ind(con_lluvia, sin_lluvia)

print("=== PRUEBA T DE STUDENT ===")
print(f"Estadístico t:  {estadistico_t:.4f}")
print(f"p-value:        {p_value:.6f}")
print()

# Interpretación con alfa = 0.05
alfa = 0.05
if p_value < alfa:
    print(f"Resultado: p ({p_value:.4f}) < alfa ({alfa})")
    print("=> Rechazamos H0.")
    print("=> La diferencia de retrasos entre días con y sin lluvia ES estadísticamente significativa.")
    print("=> Es muy improbable que esta diferencia sea producto del azar.")
else:
    print(f"Resultado: p ({p_value:.4f}) >= alfa ({alfa})")
    print("=> No rechazamos H0.")
    print("=> No hay evidencia suficiente de una diferencia real entre los grupos.")

In [ ]:
# Qué hace: calcular intervalos de confianza del 95% para cada grupo
# Por qué sirve: agrega contexto a la estimación puntual mostrando cuánta incertidumbre existe

def intervalo_confianza(datos, confianza=0.95):
    """Calcula el intervalo de confianza para la media de una muestra."""
    n = len(datos)
    media = datos.mean()
    error_estandar = stats.sem(datos)
    ic_inf, ic_sup = stats.t.interval(confianza, df=n-1, loc=media, scale=error_estandar)
    return media, ic_inf, ic_sup

media_lluvia, ic_inf_l, ic_sup_l = intervalo_confianza(con_lluvia)
media_seco,   ic_inf_s, ic_sup_s = intervalo_confianza(sin_lluvia)

print("=== INTERVALOS DE CONFIANZA DEL 95% ===")
print(f"Con lluvia:    media = {media_lluvia:.2f} min  |  IC = [{ic_inf_l:.2f}, {ic_sup_l:.2f}]")
print(f"Sin lluvia:    media = {media_seco:.2f} min   |  IC = [{ic_inf_s:.2f}, {ic_sup_s:.2f}]")
print()
print("Si los intervalos NO se superponen, la diferencia es estadísticamente significativa.")

In [ ]:
# Qué hace: prueba t comparando notas entre dos cursos del dataset de estudiantes
# Por qué sirve: practica el mismo flujo en un contexto diferente — educación

cursos = estudiantes["curso"].unique()
print("Cursos disponibles:", cursos)

# Seleccionar los dos primeros cursos disponibles
curso_a = cursos[0]
curso_b = cursos[1]

notas_a = estudiantes[estudiantes["curso"] == curso_a]["nota_final"].dropna()
notas_b = estudiantes[estudiantes["curso"] == curso_b]["nota_final"].dropna()

print(f"\n{curso_a}: media = {notas_a.mean():.2f}, n = {len(notas_a)}")
print(f"{curso_b}: media = {notas_b.mean():.2f}, n = {len(notas_b)}")

t_stat, p_val = stats.ttest_ind(notas_a, notas_b)
print(f"\nt = {t_stat:.4f}, p = {p_val:.4f}")

if p_val < 0.05:
    print(f"=> Las notas de {curso_a} y {curso_b} son significativamente diferentes (p < 0.05).")
else:
    print(f"=> No hay evidencia de diferencia significativa entre {curso_a} y {curso_b}.")

In [ ]:
# Qué hace: prueba chi-cuadrado para determinar si el curso afecta la probabilidad de aprobar
# Por qué sirve: la chi-cuadrado compara distribuciones de variables categóricas

# Crear variable binaria: aprobado/reprobado
estudiantes["aprobado"] = (estudiantes["nota_final"] >= 60).map({True: "Aprobado", False: "Reprobado"})

# Tabla de contingencia: frecuencia de cada combinación curso × resultado
tabla_contingencia = pd.crosstab(estudiantes["curso"], estudiantes["aprobado"])
print("=== TABLA DE CONTINGENCIA ===")
print(tabla_contingencia)
print()

# Prueba chi-cuadrado
chi2, p_chi, gl, frecuencias_esperadas = stats.chi2_contingency(tabla_contingencia)

print("=== PRUEBA CHI-CUADRADO ===")
print(f"Chi2:               {chi2:.4f}")
print(f"p-value:            {p_chi:.6f}")
print(f"Grados de libertad: {gl}")
print()

if p_chi < 0.05:
    print("=> Hay relación significativa entre el curso y la probabilidad de aprobar.")
else:
    print("=> No hay evidencia de relación significativa entre el curso y la aprobación.")

In [ ]:
# Qué hace: visualizar la tabla de contingencia como un heatmap de porcentajes
# Por qué sirve: facilita ver qué cursos tienen tasas de aprobación más altas o bajas

# Convertir a porcentajes por fila (por curso)
tabla_pct = tabla_contingencia.div(tabla_contingencia.sum(axis=1), axis=0) * 100

plt.figure(figsize=(7, 5))
sns.heatmap(tabla_pct, annot=True, fmt=".1f", cmap="RdYlGn",
            vmin=0, vmax=100, linewidths=0.5)
plt.title("Tasa de aprobación por curso (%)")
plt.ylabel("Curso")
plt.xlabel("Resultado")
plt.tight_layout()
plt.show()

## Resumen de la clase

| Concepto | Qué es | Para qué sirve |
|---|---|---|
| H0 | Hipótesis nula: no hay diferencia | El punto de partida de toda prueba |
| H1 | Hipótesis alternativa: sí hay diferencia | Lo que queremos comprobar |
| p-value | Probabilidad de ver este resultado si H0 fuera verdadera | Si p < 0.05, rechazamos H0 |
| Prueba t | Comparar medias de dos grupos numéricos | ¿Los promedios son iguales? |
| Chi-cuadrado | Comparar frecuencias de categorías | ¿Las distribuciones son iguales? |
| Error tipo I | Falsa alarma (rechazar H0 cuando es verdadera) | Alfa = 0.05 lo controla |
| Error tipo II | Detección fallida (no rechazar H0 cuando es falsa) | Se controla con mayor n |

**Próxima clase:** Feature engineering — prepararemos las variables de los datasets para que sean útiles como entrada a modelos de machine learning.